In [ ]:
# Step 1 - Install Dependencies

# Install core training libraries
!pip install -q transformers==4.45.0
!pip install -q datasets==3.0.1
!pip install -q peft==0.13.0
!pip install -q trl==0.11.4
!pip install -q accelerate==0.34.2

# Install evaluation libraries
!pip install -q rouge-score
!pip install -q bert-score
!pip install -q scikit-learn

# Install huggingface hub
!pip install -q huggingface_hub

print("All packages installed successfully")

In [ ]:
# Step 2 - Check GPU

import torch

# Check if GPU is available
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

# Print GPU details if available
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print("GPU memory:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")
else:
    print("No GPU found - please switch to A100 runtime")

In [ ]:
# Step 3 - Mount Google Drive

from google.colab import drive

# Mount drive
drive.mount('/content/drive')

# Define base project path
base_path = '/content/drive/MyDrive/Depixen/PROJECT_2_TEXT_TO_KG/phi35_lora'

# Define all sub folder paths
adapter_path   = base_path + '/adapter'
results_path   = base_path + '/results'
plots_path     = base_path + '/plots'
checkpoint_path = base_path + '/checkpoints'

# Create all folders if they do not exist
import os
for path in [adapter_path, results_path, plots_path, checkpoint_path]:
    os.makedirs(path, exist_ok=True)

print("Drive mounted successfully")
print("Project path:", base_path)

In [ ]:
# Step 4 - Authenticate Hugging Face

from huggingface_hub import login
from google.colab import userdata

# Load HF token from Colab secrets and login
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

print("Hugging Face authentication successful")

In [ ]:
# Step 5 - Load Dataset from Hugging Face JSONL files

from datasets import load_dataset, DatasetDict

# Define base path to processed folder on Hugging Face
base_url = "hf://datasets/BSVGK/Text_to_KG_Construction_Dataset/processed/"

# Load each pre-defined split directly from its JSONL file
print("Loading dataset...")

dataset = DatasetDict({
    'train'      : load_dataset('json', data_files=base_url + 'train.jsonl',      split='train'),
    'validation' : load_dataset('json', data_files=base_url + 'validation.jsonl', split='train'),
    'test'       : load_dataset('json', data_files=base_url + 'test.jsonl',       split='train')
})

# Print number of records per split
print("Dataset loaded successfully")
print("Train records     :", len(dataset['train']))
print("Validation records:", len(dataset['validation']))
print("Test records      :", len(dataset['test']))

# Print one sample to verify
print("\nSample record:")
print("Instruction:", dataset['train'][0]['instruction'])
print("Input      :", dataset['train'][0]['input'])
print("Output     :", dataset['train'][0]['output'])

In [ ]:
# Step 6 - Load Ontology Schema

import json
from huggingface_hub import hf_hub_download

# Download ontology schema from Hugging Face
ontology_file = hf_hub_download(
    repo_id   = "BSVGK/Text_to_KG_Construction_Dataset",
    filename  = "processed/ontology_schema.json",
    repo_type = "dataset"
)

# Load ontology schema
with open(ontology_file, 'r') as f:
    ontology = json.load(f)

# Print ontology structure
print("Ontology schema loaded successfully")
print("Ontology contents:")
print(json.dumps(ontology, indent=2))

In [ ]:
# Step 7 - Load Model and Tokenizer

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Define model name
model_name = "microsoft/Phi-3.5-mini-instruct"

# Load tokenizer
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True,
    padding_side="right"
)

# Set pad token to eos token if not defined
if tokenizer.pad_token is None:
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print("Tokenizer loaded successfully")
print("Vocab size :", tokenizer.vocab_size)
print("Pad token  :", tokenizer.pad_token)
print("EOS token  :", tokenizer.eos_token)

# Load model in float16 for A100
print("\nLoading model...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype       = torch.float16,
    device_map        = "auto",
    trust_remote_code = True,
    attn_implementation = "eager"
)

# Disable cache for training
model.config.use_cache       = False
model.config.pretraining_tp  = 1

# Print model info
total_params = sum(p.numel() for p in model.parameters())
print("Model loaded successfully")
print("Total parameters :", round(total_params / 1e9, 2), "B")
print("Memory used      :", round(torch.cuda.memory_allocated() / 1e9, 2), "GB")

In [ ]:
# Step 8 - Apply LoRA Configuration

from peft import LoraConfig, get_peft_model, TaskType

# Define LoRA configuration
lora_config = LoraConfig(
    r              = 16,
    lora_alpha     = 32,
    lora_dropout   = 0.05,
    bias           = "none",
    task_type      = TaskType.CAUSAL_LM,
    # Target attention and feed forward layers in Phi-3.5
    target_modules = [
        "q_proj", "k_proj", "v_proj",
        "o_proj", "gate_proj",
        "up_proj", "down_proj"
    ]
)

# Apply LoRA to model
model = get_peft_model(model, lora_config)

# Calculate trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params     = sum(p.numel() for p in model.parameters())
frozen_params    = total_params - trainable_params

print("LoRA applied successfully")
print("Trainable parameters :", f"{trainable_params:,}")
print("Frozen parameters    :", f"{frozen_params:,}")
print("Total parameters     :", f"{total_params:,}")
print("Trainable percentage :", round(100 * trainable_params / total_params, 4), "%")

In [ ]:
# Step 9 - Prepare Dataset and Prompt Format

# Format each record into Phi-3.5 chat format
def format_prompt(example):
    # Build prompt using Phi-3.5 native chat template
    prompt = f"""<|user|>
{example['instruction']}

Contract Description:
{example['input']}<|end|>
<|assistant|>
{example['output']}<|end|>"""

    return {"text": prompt}

# Apply formatting to all splits
print("Formatting dataset...")
train_dataset = dataset['train'].map(format_prompt,      remove_columns=dataset['train'].column_names)
val_dataset   = dataset['validation'].map(format_prompt, remove_columns=dataset['validation'].column_names)
test_dataset  = dataset['test'].map(format_prompt,       remove_columns=dataset['test'].column_names)

print("Dataset formatted successfully")
print("Train records     :", len(train_dataset))
print("Validation records:", len(val_dataset))
print("Test records      :", len(test_dataset))

# Print one formatted sample to verify
print("\nSample formatted prompt:")
print(train_dataset[0]['text'])

In [ ]:
# Step 10 - Configure Training Arguments

from trl import SFTConfig

# Define all training arguments
training_args = SFTConfig(
    # Output and saving
    output_dir          = checkpoint_path,

    # Training duration
    num_train_epochs    = 3,

    # Batch size and gradient
    per_device_train_batch_size = 4,
    per_device_eval_batch_size  = 4,
    gradient_accumulation_steps = 4,

    # Learning rate settings
    learning_rate       = 2e-4,
    lr_scheduler_type   = "cosine",
    warmup_ratio        = 0.05,
    weight_decay        = 0.01,
    max_grad_norm       = 0.3,

    # Sequence length
    max_seq_length      = 1024,

    # Logging and evaluation
    logging_steps       = 10,
    eval_steps          = 100,
    save_steps          = 100,

    # Use standard optimizer since bitsandbytes is not used
    optim               = "adamw_torch",
    fp16                = True,

    # Dataset and evaluation settings
    dataset_text_field  = "text",
    eval_strategy       = "steps",
    save_strategy       = "steps",
    load_best_model_at_end = True,
    metric_for_best_model  = "eval_loss",
    greater_is_better      = False,
    save_total_limit       = 2,

    # Disable external reporting
    report_to           = "none",
    packing             = False,
    logging_dir         = plots_path,
)

print("Training arguments configured successfully")
print("Epochs          :", training_args.num_train_epochs)
print("Batch size      :", training_args.per_device_train_batch_size)
print("Learning rate   :", training_args.learning_rate)
print("Max seq length  :", training_args.max_seq_length)
print("Optimizer       :", training_args.optim)
print("Checkpoint path :", checkpoint_path)

In [ ]:
# Step 11 - Initialize Trainer

from trl import SFTTrainer

# Initialize SFT Trainer with model, data and training arguments
trainer = SFTTrainer(
    model         = model,
    args          = training_args,
    train_dataset = train_dataset,
    eval_dataset  = val_dataset,
    tokenizer     = tokenizer,
)

# Calculate total training steps
total_steps = (len(train_dataset) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)) * training_args.num_train_epochs

print("Trainer initialized successfully")
print("Total training samples :", len(train_dataset))
print("Total training steps   :", total_steps)
print("Eval every             :", training_args.eval_steps, "steps")
print("Save every             :", training_args.save_steps, "steps")
print("Best model metric      :", training_args.metric_for_best_model)

In [ ]:
# Step 12 - Start Training

import time

# Record start time
start_time = time.time()

print("Starting training...")
print("Model          : Phi-3.5 Mini Instruct")
print("Method         : LoRA")
print("Epochs         : 3")
print("Total steps    : 1212")
print("Logging every  : 10 steps")
print("Eval every     : 100 steps")
print("-" * 50)

# Start training
train_result = trainer.train()

# Calculate total training time
end_time     = time.time()
total_time   = round((end_time - start_time) / 60, 2)

# Print training summary
print("-" * 50)
print("Training completed successfully")
print("Total steps    :", train_result.global_step)
print("Training loss  :", round(train_result.training_loss, 4))
print("Training time  :", total_time, "minutes")

In [ ]:
# Step 13 - Save Training Plots to Drive

import matplotlib.pyplot as plt

# Extract loss values from trainer log history
train_losses = [(x['step'], x['loss'])      for x in trainer.state.log_history if 'loss' in x]
eval_losses  = [(x['step'], x['eval_loss']) for x in trainer.state.log_history if 'eval_loss' in x]

# Separate steps and loss values
train_steps, train_loss_values = zip(*train_losses)
eval_steps,  eval_loss_values  = zip(*eval_losses)

# Plot train loss vs eval loss
plt.figure(figsize=(10, 5))
plt.plot(train_steps, train_loss_values, label='Train Loss',      color='steelblue')
plt.plot(eval_steps,  eval_loss_values,  label='Validation Loss', color='orange', marker='o')
plt.xlabel('Training Steps')
plt.ylabel('Loss')
plt.title('Phi-3.5 Mini + LoRA - Training vs Validation Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

# Save to Drive
loss_plot_path = plots_path + '/loss_curve.png'
plt.savefig(loss_plot_path, dpi=150)
plt.show()
print("Loss curve saved to:", loss_plot_path)

# Plot learning rate schedule
lr_values = [(x['step'], x['learning_rate']) for x in trainer.state.log_history if 'learning_rate' in x]
lr_steps, lr_vals = zip(*lr_values)

plt.figure(figsize=(10, 4))
plt.plot(lr_steps, lr_vals, color='green')
plt.xlabel('Training Steps')
plt.ylabel('Learning Rate')
plt.title('Phi-3.5 Mini + LoRA - Learning Rate Schedule')
plt.grid(True, alpha=0.3)
plt.tight_layout()

# Save to Drive
lr_plot_path = plots_path + '/lr_schedule.png'
plt.savefig(lr_plot_path, dpi=150)
plt.show()
print("Learning rate schedule saved to:", lr_plot_path)

In [ ]:
# Step 14 - Save LoRA Adapter to Drive

import os

# Save LoRA adapter weights to Drive
print("Saving LoRA adapter to Drive...")
trainer.model.save_pretrained(adapter_path)

# Save tokenizer to same folder
print("Saving tokenizer to Drive...")
tokenizer.save_pretrained(adapter_path)

# Print saved files and sizes
print("\nAdapter saved successfully")
print("Adapter path:", adapter_path)
print("\nSaved files:")
for f in os.listdir(adapter_path):
    size = os.path.getsize(os.path.join(adapter_path, f)) / 1e6
    print(f"  {f} : {round(size, 2)} MB")

In [ ]:
# Step 15a - Test Inference on One Record to Verify Model Output

import torch

# Fix padding side for decoder only model
tokenizer.padding_side = "left"

# Set model to evaluation mode
model.eval()

# Pick one sample from test set
sample = dataset['test'][0]

# Build prompt
prompt = f"""<|user|>
{sample['instruction']}

Contract Description:
{sample['input']}<|end|>
<|assistant|>
"""

# Tokenize
inputs = tokenizer(
    prompt,
    return_tensors = "pt",
    truncation     = True,
    max_length     = 800
).to(model.device)

# Generate prediction
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens     = 200,
        do_sample          = False,
        repetition_penalty = 1.1,
        pad_token_id       = tokenizer.pad_token_id,
        eos_token_id       = tokenizer.eos_token_id,
    )

# Decode output
input_length = inputs.input_ids.shape[1]
predicted    = tokenizer.decode(
    outputs[0][input_length:],
    skip_special_tokens = True
).strip()

# Print results
print("Input:")
print(sample['input'])
print("\nPredicted Triples:")
print(predicted)
print("\nGround Truth Triples:")
print(sample['output'])

In [ ]:
# Step 15b - Run Inference on Full Test Set using Batch Inference

import torch

# Fix padding side for decoder only model
tokenizer.padding_side = "left"

# Set model to evaluation mode
model.eval()

# Batch size for inference
INFERENCE_BATCH_SIZE = 8

# Build prompt for each sample
def build_prompt(sample):
    return f"""<|user|>
{sample['instruction']}

Contract Description:
{sample['input']}<|end|>
<|assistant|>
"""

# Prepare all prompts and ground truths from full test set
print("Building prompts...")
prompts       = [build_prompt(s) for s in dataset['test']]
ground_truths = [s['output'] for s in dataset['test']]
input_texts   = [s['input']  for s in dataset['test']]

total_samples = len(prompts)
print("Total test samples:", total_samples)
print("Running batch inference...")
print("Batch size        :", INFERENCE_BATCH_SIZE)
print("-" * 40)

predictions = []

for i in range(0, total_samples, INFERENCE_BATCH_SIZE):
    # Get current batch of prompts
    batch_prompts = prompts[i : i + INFERENCE_BATCH_SIZE]

    # Tokenize batch with left padding
    inputs = tokenizer(
        batch_prompts,
        return_tensors = "pt",
        truncation     = True,
        padding        = True,
        max_length     = 800
    ).to(model.device)

    # Generate predictions for batch
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens     = 200,
            do_sample          = False,
            repetition_penalty = 1.1,
            pad_token_id       = tokenizer.pad_token_id,
            eos_token_id       = tokenizer.eos_token_id,
        )

    # Decode each output in the batch
    input_length = inputs.input_ids.shape[1]
    for output in outputs:
        generated = tokenizer.decode(
            output[input_length:],
            skip_special_tokens = True
        ).strip()
        predictions.append(generated)

    # Track and print progress every 100 samples
    processed = min(i + INFERENCE_BATCH_SIZE, total_samples)
    if processed % 100 < INFERENCE_BATCH_SIZE or processed == total_samples:
        print(f"Processed {processed} / {total_samples} samples")

print("-" * 40)
print("Inference completed successfully")
print("Total predictions:", len(predictions))

# Show one sample to verify
print("\nSample prediction:")
print("Input        :", input_texts[0])
print("\nPredicted    :", predictions[0])
print("\nGround truth :", ground_truths[0])

In [ ]:
# Step 16 - Compute All Evaluation Metrics

import re
import numpy as np
from rouge_score import rouge_scorer
from bert_score import score as bert_score

# Define valid relations from ontology for hallucination check
valid_relations = [
    "rdf:type",
    "hasBuyer",
    "hasSupplier",
    "hasContractValue",
    "hasAwardDate",
    "hasCPVCode",
    "hasCPVDescription",
    "hasLocation"
]

# Parse triples from output text into a list of tuples
def parse_triples(text):
    triples = []
    for line in text.strip().split('\n'):
        line = line.strip()
        if line.startswith('(') and line.endswith(')'):
            # Remove brackets and split by comma
            content = line[1:-1].split(',')
            if len(content) == 3:
                triple = tuple(part.strip() for part in content)
                triples.append(triple)
    return triples

# Compute precision recall and f1 for one sample
def compute_prf(pred_triples, gt_triples):
    pred_set = set(pred_triples)
    gt_set   = set(gt_triples)

    if not pred_set and not gt_set:
        return 1.0, 1.0, 1.0
    if not pred_set or not gt_set:
        return 0.0, 0.0, 0.0

    tp        = len(pred_set & gt_set)
    precision = tp / len(pred_set)
    recall    = tp / len(gt_set)
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    return precision, recall, f1

# Check hallucination level 1 - relation not in ontology
def check_hallucination_level1(pred_triples):
    hallucinated = 0
    total        = len(pred_triples)
    for triple in pred_triples:
        if triple[1] not in valid_relations:
            hallucinated += 1
    return hallucinated / total if total > 0 else 0.0

# Check hallucination level 2 - entity value not in input text
def check_hallucination_level2(pred_triples, input_text):
    hallucinated = 0
    total        = len(pred_triples)
    for triple in pred_triples:
        # Check if subject or object appears in input text
        subject = triple[0].lower()
        obj     = triple[2].lower()
        input_lower = input_text.lower()
        if subject not in input_lower and obj not in input_lower:
            hallucinated += 1
    return hallucinated / total if total > 0 else 0.0

# Compute all metrics
print("Computing evaluation metrics...")
print("Total samples:", len(predictions))
print("-" * 40)

precision_scores   = []
recall_scores      = []
f1_scores          = []
hallucination_l1   = []
hallucination_l2   = []

for i in range(len(predictions)):
    pred_triples = parse_triples(predictions[i])
    gt_triples   = parse_triples(ground_truths[i])

    # Compute precision recall f1
    p, r, f1 = compute_prf(pred_triples, gt_triples)
    precision_scores.append(p)
    recall_scores.append(r)
    f1_scores.append(f1)

    # Compute hallucination level 1
    h1 = check_hallucination_level1(pred_triples)
    hallucination_l1.append(h1)

    # Compute hallucination level 2
    h2 = check_hallucination_level2(pred_triples, input_texts[i])
    hallucination_l2.append(h2)

print("Precision, Recall, F1 and Hallucination computed")

# Compute ROUGE-L scores
print("Computing ROUGE-L scores...")
scorer       = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
rouge_scores = []
for pred, gt in zip(predictions, ground_truths):
    score = scorer.score(gt, pred)
    rouge_scores.append(score['rougeL'].fmeasure)

print("ROUGE-L computed")

# Compute BERTScore on full output string
print("Computing BERTScore...")
P, R, F1_bert = bert_score(
    predictions,
    ground_truths,
    lang          = "en",
    verbose       = False,
    device        = "cuda"
)
bert_scores = F1_bert.tolist()
print("BERTScore computed")

# Print all results
print("-" * 40)
print("EVALUATION RESULTS - Phi-3.5 Mini + LoRA")
print("-" * 40)
print("Primary Metrics:")
print("  F1 Score  :", round(np.mean(f1_scores),  4))
print("  Recall    :", round(np.mean(recall_scores), 4))
print("\nSecondary Metrics:")
print("  Precision          :", round(np.mean(precision_scores),  4))
print("  ROUGE-L            :", round(np.mean(rouge_scores),      4))
print("  BERTScore          :", round(np.mean(bert_scores),       4))
print("  Hallucination L1   :", round(np.mean(hallucination_l1),  4))
print("  Hallucination L2   :", round(np.mean(hallucination_l2),  4))
print("-" * 40)

In [ ]:
# Step 17 - Save Results to Drive

import pandas as pd
import numpy as np
import os

# Compile all metrics into a dictionary
results = {
    'Model'            : 'Phi-3.5 Mini Instruct',
    'Method'           : 'LoRA',
    'LoRA Rank'        : 16,
    'LoRA Alpha'       : 32,
    'Epochs'           : 3,
    'Learning Rate'    : 2e-4,
    'Train Samples'    : len(dataset['train']),
    'Test Samples'     : len(predictions),
    'F1 Score'         : round(np.mean(f1_scores),         4),
    'Recall'           : round(np.mean(recall_scores),     4),
    'Precision'        : round(np.mean(precision_scores),  4),
    'ROUGE-L'          : round(np.mean(rouge_scores),      4),
    'BERTScore'        : round(np.mean(bert_scores),       4),
    'Hallucination L1' : round(np.mean(hallucination_l1),  4),
    'Hallucination L2' : round(np.mean(hallucination_l2),  4),
    'Training Loss'    : round(train_result.training_loss, 4),
    'Training Time'    : '26.01 minutes',
}

# Convert to dataframe
df_results = pd.DataFrame([results])

# Save to Drive
results_csv_path = results_path + '/metrics.csv'
df_results.to_csv(results_csv_path, index=False)

print("Results saved successfully")
print("Results path:", results_csv_path)
print("\nFinal Results Table:")
print("-" * 40)
for key, value in results.items():
    print(f"  {key:<20} : {value}")
print("-" * 40)

**Running the full test inference from saved adapters and save the predictions and deploy the model in hugging face**

In [ ]:
# Step 8 - Load Saved LoRA Adapter from Drive

from peft import PeftModel

print("Loading saved LoRA adapter from Drive...")
model = PeftModel.from_pretrained(
    model,
    adapter_path,
    is_trainable = False
)
model.eval()
tokenizer.padding_side = "left"
print("Adapter loaded successfully")
print("Memory used:", round(torch.cuda.memory_allocated() / 1e9, 2), "GB")

In [ ]:
# Step 9 - Run Full Batch Inference on All Test Records

import torch

model.eval()
INFERENCE_BATCH_SIZE = 8

def build_prompt(sample):
    return f"""<|user|>
{sample['instruction']}

Contract Description:
{sample['input']}<|end|>
<|assistant|>
"""

print("Building prompts...")
prompts       = [build_prompt(s) for s in dataset['test']]
ground_truths = [s['output'] for s in dataset['test']]
input_texts   = [s['input']  for s in dataset['test']]

print("Total test samples:", len(prompts))
print("Batch size        :", INFERENCE_BATCH_SIZE)
print("-" * 40)

predictions = []

for i in range(0, len(prompts), INFERENCE_BATCH_SIZE):
    batch_prompts = prompts[i : i + INFERENCE_BATCH_SIZE]

    inputs = tokenizer(
        batch_prompts,
        return_tensors = "pt",
        truncation     = True,
        padding        = True,
        max_length     = 800
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens     = 200,
            do_sample          = False,
            repetition_penalty = 1.1,
            pad_token_id       = tokenizer.pad_token_id,
            eos_token_id       = tokenizer.eos_token_id,
        )

    input_length = inputs.input_ids.shape[1]
    for output in outputs:
        generated = tokenizer.decode(
            output[input_length:],
            skip_special_tokens = True
        ).strip()
        predictions.append(generated)

    completed = min(i + INFERENCE_BATCH_SIZE, len(prompts))
    print(f"Completed {completed} / {len(prompts)} records")

print("\nInference completed")
print("Total predictions:", len(predictions))

In [ ]:
# Step 10 - Compute All Evaluation Metrics

import numpy as np
from rouge_score import rouge_scorer
from bert_score import score as bert_score_fn

valid_relations = [
    "rdf:type", "hasBuyer", "hasSupplier", "hasContractValue",
    "hasAwardDate", "hasCPVCode", "hasCPVDescription", "hasLocation"
]

def parse_triples(text):
    triples = []
    for line in text.strip().split('\n'):
        line = line.strip()
        if line.startswith('(') and line.endswith(')'):
            content = line[1:-1].split(',')
            if len(content) == 3:
                triple = tuple(part.strip() for part in content)
                triples.append(triple)
    return triples

def compute_prf(pred_triples, gt_triples):
    pred_set = set(pred_triples)
    gt_set   = set(gt_triples)
    if not pred_set and not gt_set:
        return 1.0, 1.0, 1.0
    if not pred_set or not gt_set:
        return 0.0, 0.0, 0.0
    tp        = len(pred_set & gt_set)
    precision = tp / len(pred_set)
    recall    = tp / len(gt_set)
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return precision, recall, f1

def check_hallucination_l1(pred_triples):
    if not pred_triples:
        return 0.0
    hallucinated = sum(1 for t in pred_triples if t[1] not in valid_relations)
    return hallucinated / len(pred_triples)

def check_hallucination_l2(pred_triples, input_text):
    if not pred_triples:
        return 0.0
    input_lower  = input_text.lower()
    hallucinated = sum(1 for t in pred_triples if t[0].lower() not in input_lower and t[2].lower() not in input_lower)
    return hallucinated / len(pred_triples)

print("Computing metrics...")

precision_scores = []
recall_scores    = []
f1_scores        = []
rouge_scores     = []
hallucination_l1 = []
hallucination_l2 = []

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

for i in range(len(predictions)):
    pred_triples = parse_triples(predictions[i])
    gt_triples   = parse_triples(ground_truths[i])

    p, r, f1 = compute_prf(pred_triples, gt_triples)
    precision_scores.append(p)
    recall_scores.append(r)
    f1_scores.append(f1)

    rouge_l = scorer.score(ground_truths[i], predictions[i])['rougeL'].fmeasure
    rouge_scores.append(rouge_l)

    h1 = check_hallucination_l1(pred_triples)
    h2 = check_hallucination_l2(pred_triples, input_texts[i])
    hallucination_l1.append(h1)
    hallucination_l2.append(h2)

print("Computing BERTScore...")
P, R, F = bert_score_fn(
    predictions,
    ground_truths,
    lang       = "en",
    model_type = "distilbert-base-uncased",
    verbose    = False
)
bert_scores = F.tolist()

print("\nFinal Evaluation Results")
print("-" * 40)
print("F1 Score         :", round(np.mean(f1_scores), 4))
print("Recall           :", round(np.mean(recall_scores), 4))
print("Precision        :", round(np.mean(precision_scores), 4))
print("ROUGE-L          :", round(np.mean(rouge_scores), 4))
print("BERTScore        :", round(np.mean(bert_scores), 4))
print("Hallucination L1 :", round(np.mean(hallucination_l1), 4))
print("Hallucination L2 :", round(np.mean(hallucination_l2), 4))

In [ ]:
# Step 11 - Save All Results to Drive

import pandas as pd
import numpy as np
import os

# Save predictions with per-record metrics
df_predictions = pd.DataFrame({
    'record_id'        : list(range(1, len(predictions) + 1)),
    'input'            : input_texts,
    'ground_truth'     : ground_truths,
    'prediction'       : predictions,
    'precision'        : [round(p, 4) for p in precision_scores],
    'recall'           : [round(r, 4) for r in recall_scores],
    'f1'               : [round(f, 4) for f in f1_scores],
    'rouge_l'          : [round(r, 4) for r in rouge_scores],
    'bert_score'       : [round(b, 4) for b in bert_scores],
    'hallucination_l1' : [round(h, 4) for h in hallucination_l1],
    'hallucination_l2' : [round(h, 4) for h in hallucination_l2],
})

predictions_csv_path = results_path + '/predictions.csv'
df_predictions.to_csv(predictions_csv_path, index=False)
print("Predictions CSV saved to:", predictions_csv_path)
print("Total records saved     :", len(df_predictions))

# Save summary metrics
results = {
    'model'            : 'Phi-3.5 Mini Instruct',
    'method'           : 'LoRA',
    'lora_rank'        : 16,
    'lora_alpha'       : 32,
    'epochs'           : 3,
    'learning_rate'    : 2e-4,
    'train_samples'    : len(dataset['train']),
    'test_samples'     : len(predictions),
    'f1_score'         : round(np.mean(f1_scores), 4),
    'recall'           : round(np.mean(recall_scores), 4),
    'precision'        : round(np.mean(precision_scores), 4),
    'rouge-l'          : round(np.mean(rouge_scores), 4),
    'bertscore'        : round(np.mean(bert_scores), 4),
    'hallucination_l1' : round(np.mean(hallucination_l1), 4),
    'hallucination_l2' : round(np.mean(hallucination_l2), 4),
    'training_loss'    : 0.3008,
    'training_time'    : 26.01,
}

df_metrics = pd.DataFrame([results])
metrics_csv_path = results_path + '/metrics.csv'
df_metrics.to_csv(metrics_csv_path, index=False)
print("Metrics CSV saved to    :", metrics_csv_path)

# Verify all files saved
print("\nAll files in results folder:")
for f in os.listdir(results_path):
    size = os.path.getsize(os.path.join(results_path, f)) / 1e6
    print(f"  {f} : {round(size, 2)} MB")

In [ ]:
# Step 12 - Save Merged Model to Drive and Push to HuggingFace

import os
from huggingface_hub import HfApi

HF_USERNAME      = "BSVGK"
ADAPTER_REPO     = f"{HF_USERNAME}/phi35-mini-lora-text2kg-adapter"
MERGED_REPO      = f"{HF_USERNAME}/phi35-mini-lora-text2kg-merged"
merged_drive_path = base_path + '/merged'
merged_local_path = '/content/phi35_lora_merged'

os.makedirs(merged_drive_path, exist_ok=True)

api = HfApi()

# Push LoRA adapter only to HuggingFace
print("Pushing LoRA adapter to HuggingFace...")
api.create_repo(repo_id=ADAPTER_REPO, token=hf_token, private=True, exist_ok=True)
model.push_to_hub(ADAPTER_REPO, token=hf_token)
tokenizer.push_to_hub(ADAPTER_REPO, token=hf_token)
print(f"Adapter pushed to: https://huggingface.co/{ADAPTER_REPO}")

# Merge adapter into base model
print("\nMerging adapter into base model...")
merged_model = model.merge_and_unload()
merged_model.eval()
print("Merge complete. Memory:", round(torch.cuda.memory_allocated() / 1e9, 2), "GB")

# Fix tied_weights_keys bug
for name, module in merged_model.named_modules():
    if hasattr(module, '_tied_weights_keys'):
        if isinstance(module._tied_weights_keys, list):
            module._tied_weights_keys = {}

# Save merged model to Drive
print("\nSaving merged model to Drive...")
merged_model.save_pretrained(merged_drive_path)
tokenizer.save_pretrained(merged_drive_path)
print("Merged model saved to Drive:", merged_drive_path)
print("Files saved:")
for f in os.listdir(merged_drive_path):
    size = os.path.getsize(os.path.join(merged_drive_path, f)) / 1e6
    print(f"  {f} : {round(size, 2)} MB")

# Save merged model locally then push to HuggingFace
print("\nSaving merged model locally for HF upload...")
os.makedirs(merged_local_path, exist_ok=True)
merged_model.save_pretrained(merged_local_path)
tokenizer.save_pretrained(merged_local_path)

print(f"\nPushing merged model to HuggingFace: {MERGED_REPO}")
api.create_repo(repo_id=MERGED_REPO, token=hf_token, private=True, exist_ok=True)
api.upload_folder(folder_path=merged_local_path, repo_id=MERGED_REPO, token=hf_token)
print(f"Merged model pushed to: https://huggingface.co/{MERGED_REPO}")

# Verify all Drive folders
print("\nAll saved locations:")
print("Adapter (Drive)      :", adapter_path)
print("Merged model (Drive) :", merged_drive_path)
print("Predictions CSV      :", predictions_csv_path)
print("Metrics CSV          :", metrics_csv_path)
print("Adapter (HF)         :", f"https://huggingface.co/{ADAPTER_REPO}")
print("Merged model (HF)    :", f"https://huggingface.co/{MERGED_REPO}")